<a href="https://colab.research.google.com/github/bvsvntv/ai-ml-learning/blob/main/lang-graph/prompt_chaining_workflow.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -U langchain-openai

In [2]:
from google.colab import userdata
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI
from typing import TypedDict

In [3]:
apiKey = userdata.get('OPENAI_API_KEY')
model = ChatOpenAI(api_key = apiKey, model = "gpt-6-luna")

In [13]:
# state
class BlogState:
  title: str
  outline: str
  content: str
  score: str

In [14]:
def create_outline(state: BlogState) -> BlogState:
  title = state["title"]
  prompt = f"Generate a detailed outline on the topic - {title}"

  outline = model.invoke(prompt).content

  state["outline"] = outline

  return state

In [15]:
def create_blog(state: BlogState) -> BlogState:
  title = state["title"]
  outline = state["outline"]

  prompt = f"Write a detailed blog on topic - {title} using outlines: \n {outline}"

  content = model.invoke(prompt).content

  state["content"] = content

  return state

In [22]:
def evaluate_blog(state: BlogState) -> BlogState:
  outline = state["outline"]
  content = state["content"]

  prompt = f"Based on this outline:\n{outline}\n\nRate my blog content out of 10:\n{content}\n\nReturn only the score (e.g., 8/10)."
  score = model.invoke(prompt).content

  state["score"] = score

  return state

In [23]:
# graph
graph = StateGraph(BlogState)

# add nodes
graph.add_node("create_outline", create_outline)
graph.add_node("create_blog", create_blog)
graph.add_node("evaluate_blog", evaluate_blog)

# add edges
graph.add_edge(START, "create_outline")
graph.add_edge("create_outline", "create_blog")
graph.add_edge("create_blog", "evaluate_blog")
graph.add_edge("evaluate_blog", END)

# compile
workflow = graph.compile()

In [24]:
# execute
initial_state = {"title": "Building LLM agents."}

final_state = workflow.invoke(initial_state)

print("outlines:\n", final_state["outline"])
print("blog:\n", final_state["content"])
print("score: ", final_state["score"])

outlines:
 # Building LLM Agents: A Detailed Outline

## 1. Foundations

### 1.1 What Is an LLM Agent?
- A software system that uses a language model to pursue a goal through one or more steps.
- Common capabilities:
  - Interpret instructions and context
  - Choose actions
  - Use tools or external services
  - Observe results and adapt
  - Stop when the goal is complete or a limit is reached

### 1.2 Agents, Chatbots, and Workflows
- **Chatbot:** Primarily responds to user input.
- **Workflow:** Executes a predefined sequence of steps.
- **Agent:** Selects or adjusts steps in response to the task and intermediate results.
- Hybrid designs:
  - Deterministic workflows with LLM-powered steps
  - Agents operating within tightly defined workflows

### 1.3 When to Use an Agent
- Tasks require multiple steps or tool calls.
- The right sequence of actions depends on intermediate results.
- The system must interact with external information or services.
- The task can be bounded, observed, a